# Safety-Relevant Features

Safety-oriented mechanistic interpretability: refusal direction identification,
knowledge localization, deception detection signatures, alignment circuit
analysis, and safety feature monitoring.

References:
- Arditi et al. (2024) "Refusal in Language Models Is Mediated by a Single Direction"
- Marks et al. (2024) "The Geometry of Truth"
- Li et al. (2024) "Inference-Time Intervention"

## Why This Matters

Safety-relevant relevant features connects interpretability directly to AI safety by identifying internal representations of safety-critical behaviors. Locating refusal directions, deception signatures, and alignment-relevant circuits enables monitoring and intervention for safer AI systems.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Zou et al. (2023) "Representation Engineering"](https://arxiv.org/abs/2310.01405) — Reading and controlling model behavior via representations

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.safety_relevant_features import (
    refusal_direction_analysis,
    knowledge_localization,
    deception_detection_signatures,
    alignment_circuit_analysis,
    safety_feature_monitoring,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([0, 5, 10, 15, 20, 25, 30])
metric_fn = lambda logits: float(logits[-1, 0] - logits[-1, 1])
print("Model ready")

## Refusal Direction Analysis

Find the direction in residual stream space that separates
compliant vs refused inputs.

In [ ]:
# Simulate compliant vs refused inputs
compliant = [jnp.array([1, 2, 3, 4, 5, 6, 7]), jnp.array([10, 20, 30, 40, 50, 60, 70])]
refused = [jnp.array([5, 15, 25, 35, 45, 55, 65]), jnp.array([8, 18, 28, 38, 48, 58, 68])]

result = refusal_direction_analysis(model, compliant, refused)
print(f"Direction norm: {result['direction_norm']:.4f}")
print(f"Separation score: {result['separation_score']:.4f}")
print(f"Layer analyzed: {result['layer_index']}")
print(f"Compliant projections: {[f'{p:.3f}' for p in result['compliant_projections']]}")
print(f"Refused projections: {[f'{p:.3f}' for p in result['refused_projections']]}")

## Knowledge Localization

Find which layers and components are critical for a specific piece of knowledge.

In [ ]:
result = knowledge_localization(model, tokens, [0, 1, 2], metric_fn)
print(f"Critical layers: {result['critical_layers']}")
print(f"Knowledge concentrated: {result['knowledge_concentrated']}")
for i in range(cfg.n_layers):
    print(f"  Layer {i}: importance={result['layer_importance'][i]:.4f}, "
          f"attn_ratio={result['attn_vs_mlp'][i]:.2f}")

## Deception Detection Signatures

Compare internal representations between different input types.

In [ ]:
honest = jnp.array([1, 2, 3, 4, 5, 6, 7])
deceptive = jnp.array([10, 20, 30, 40, 50, 60, 70])

result = deception_detection_signatures(model, honest, deceptive)
print(f"Divergence onset layer: {result['divergence_onset_layer']}")
print(f"Max divergence layer: {result['max_divergence_layer']}")
for i in range(cfg.n_layers):
    print(f"  Layer {i}: divergence={result['layer_divergence'][i]:.4f}, "
          f"cosine={result['cosine_similarity'][i]:.4f}")

## Alignment Circuit Analysis

Find heads that contribute to behavior vs safety, and identify
conflicts and synergies.

In [ ]:
safety_fn = lambda logits: float(logits[-1, 2] - logits[-1, 3])
result = alignment_circuit_analysis(model, tokens, metric_fn, safety_fn)
print(f"Overlap score: {result['overlap_score']:.4f}")
print(f"Conflict heads: {result['conflict_heads']}")
print(f"Synergy heads: {result['synergy_heads']}")

## Safety Feature Monitoring

Monitor activations projected onto known safety-relevant directions.

In [ ]:
# Create some reference directions
directions = {
    "refusal": np.random.randn(cfg.d_model),
    "helpfulness": np.random.randn(cfg.d_model),
}

result = safety_feature_monitoring(model, tokens, reference_directions=directions)
print(f"Activation norm: {result['activation_norm']:.4f}")
print(f"Anomaly score: {result['anomaly_score']:.4f}")
print(f"\nProjections:")
for name, proj in result['projections'].items():
    print(f"  {name}: {proj:.4f}")
print(f"\nTop active dimensions:")
for dim, val in result['top_active_dimensions'][:5]:
    print(f"  dim {dim}: {val:+.4f}")